In [1]:
import os

from sklearn.cluster import HDBSCAN
from umap import UMAP

from entity import DatasetManager, TextDatasets, DatasetFactory
from utils import evaluate, random_subset
import optuna

os.chdir("..")

/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
umap = UMAP(n_components=5, metric="cosine", random_state=42)
dataset_manager = DatasetManager("intfloat/multilingual-e5-large")
dataset = dataset_manager.get(TextDatasets.WIKIPEDIA_CHEMISTRY_TOPIC_CLUSTERING)
x, y = random_subset(dataset.embeddings, dataset.labels, 1000, seed=2)
x_red = umap.fit_transform(x)

/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [9]:
hdbscan = HDBSCAN(min_cluster_size=20)
res = evaluate(hdbscan, "clustrec_covid", x_red, y)
res


/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


ExperimentResult(clusterer_name='HDBSCAN', dataset_name='clustrec_covid', dim=5, nmi=0.8555185776680229, ari=0.7768436082496286, ami=0.8529696766417707, v_measure=0.8555185776680229, noise_fraction=0.112, homogeneity=0.8424126360363041, completeness=0.869038758584708, fowlkes_mallows=0.8056185629981606, silhouette=0.6242760419845581, calinski_harabasz=1916.4801433703094, davies_bouldin=0.5335637586550611, n_clusters=9, n_true_clusters=9, running_time_s=0.007323249999899417, params={'algorithm': 'auto', 'allow_single_cluster': False, 'alpha': 1.0, 'cluster_selection_epsilon': 0.0, 'cluster_selection_method': 'eom', 'copy': 'warn', 'leaf_size': 40, 'max_cluster_size': None, 'metric': 'euclidean', 'metric_params': None, 'min_cluster_size': 20, 'min_samples': None, 'n_jobs': None, 'store_centers': None})

In [4]:
def find_hdbscan_params(dataset_id):
    dataset = dataset_manager.get(dataset_id)
    x, y = random_subset(dataset.embeddings, dataset.labels, min(1000, dataset.labels.shape[0]), seed=2)
    x_red = umap.fit_transform(x)


    def search(trial):
        min_cluster_size = trial.suggest_int("min_cluster_size", 2, 100)
        hdbscan = HDBSCAN(min_cluster_size=min_cluster_size)
        res = evaluate(hdbscan, "clustrec_covid", x_red, y)
        cluster_diff = res.n_clusters - res.n_true_clusters
        if cluster_diff < 0:
            cluster_diff = cluster_diff * -1
        cluster_diff = cluster_diff / res.n_true_clusters
        return res.v_measure * (1 - res.noise_fraction) * (1 - cluster_diff)


    study = optuna.create_study(direction="maximize")
    study.optimize(search, n_trials=100, timeout=60)
    return study.best_params

import json

config = json.load(open("config.json"))
results = []

for dataset_name in config["datasets"]:
    dataset_id = DatasetFactory.from_string(dataset_name)
    best_params = find_hdbscan_params(dataset_id)
    print(f"Best params for {dataset_name}: {best_params}")

    hdbscan = HDBSCAN(min_cluster_size=best_params["min_cluster_size"])
    res = evaluate(hdbscan, "clustrec_covid", x_red, y)
    results.append({"dataset": dataset_name, "best_params": best_params, "v_measure": res.v_measure, "noise_fraction": res.noise_fraction, "n_clusters": res.n_clusters, "n_true_clusters": res.n_true_clusters, "scaled_v_measure": res.v_measure * (1 - res.noise_fraction)})

[I 2026-09-15 10:04:52,329] A new study created in memory with name: no-name-0713a56b-4f29-4609-9443-4ab0f4d281f9
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-15 10:04:52,353] Trial 0 finished with value: 0.0040990066819050885 and parameters: {'min_cluster_size': 100}. Best is trial 0 with value: 0.0040990066819050885.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-15 10:04:52,371] Trial 1 finished with value: 0.0048958535860524005 and parameters: {'min_cluster_size': 28}. Best is trial 1 with value: 0.0048958535860524005.
/Users/floria

Best params for Uri-ka/ClusTREC-Covid: {'min_cluster_size': 4}


[I 2026-09-15 10:04:55,858] A new study created in memory with name: no-name-5f245818-81c9-448c-a409-65c084eb3db9
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/var/folders/3g/b3s59vc97cq7dg4yvgxr5mtc0000gp/T/ipykernel_31619/2711210223.py:10: UserWarning: Internal clustering metrics undefined for clustrec_covid/HDBSCAN: n_clusters=0, n_samples=0. Setting silhouette/calinski_harabasz/davies_bouldin to NaN.
  res = evaluate(hdbscan, "clustrec_covid", x_red, y)
[I 2026-09-15 10:04:55,874] Trial 0 finished with value: 0.0 and parameters: {'min_cluster_size': 93}. Best is trial 0 with value: 0.0.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to 

Best params for mteb/biorxiv-clustering-p2p: {'min_cluster_size': 5}


[I 2026-09-15 10:04:59,291] A new study created in memory with name: no-name-04b67ad3-b2f5-4444-a9e7-7aafa59b67d0
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-15 10:04:59,307] Trial 0 finished with value: 0.023895341539440194 and parameters: {'min_cluster_size': 10}. Best is trial 0 with value: 0.023895341539440194.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-15 10:04:59,327] Trial 1 finished with value: 0.0028533560744658537 and parameters: {'min_cluster_size': 70}. Best is trial 0 with value: 0.023895341539440194.
/Users/florian/de

...

Best params for mteb/reddit-clustering: {'min_cluster_size': 4}


[I 2026-09-15 10:06:11,032] A new study created in memory with name: no-name-bd7184cb-a60c-4faf-b8f7-15c7f0c6d1ed
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-15 10:06:11,052] Trial 0 finished with value: 0.00156632625055365 and parameters: {'min_cluster_size': 48}. Best is trial 0 with value: 0.00156632625055365.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-15 10:06:11,074] Trial 1 finished with value: 0.0015375109933448196 and parameters: {'min_cluster_size': 81}. Best is trial 0 with value: 0.00156632625055365.
/Users/florian/dev/u

Best params for mteb/reddit-clustering-p2p: {'min_cluster_size': 2}


[I 2026-09-15 10:06:14,900] A new study created in memory with name: no-name-2709feff-263b-4ea6-a90d-4152bb2a6774
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-15 10:06:14,916] Trial 0 finished with value: 0.48005293348739836 and parameters: {'min_cluster_size': 23}. Best is trial 0 with value: 0.48005293348739836.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-15 10:06:14,932] Trial 1 finished with value: 0.0 and parameters: {'min_cluster_size': 12}. Best is trial 0 with value: 0.48005293348739836.
/Users/florian/dev/uni/python/vMF-HAC/

Best params for mteb/DigikalamagClustering: {'min_cluster_size': 23}


In [5]:
import pandas as pd
pd.DataFrame(results).to_csv("hdbscan_parameters.csv", index=False)

In [6]:
hdbscan = HDBSCAN(min_cluster_size=study.best_params["min_cluster_size"])
res = evaluate(hdbscan, "clustrec_covid", x_red, y)
res

NameError: name 'study' is not defined

In [ ]:
(1-res.noise_fraction) * res.v_measure